
# `final_EGN.py` 검증 보고서 — Carena 2014 Fig. 1 / 3 / 6 / 8

**목적:** `final_EGN.py`를 Carena et al. (2014), *EGN model of non-linear fiber propagation*의 Fig. 1, 3, 6, 8과 직접 비교하여
1) paper vs code 수치 오차를 계산하고, 2) GN baseline과 EGN correction을 분리 평가하며, 3) 다른 실험/시뮬레이션에 사용할 수 있는지를 판단합니다.

### 핵심 결론
- **GN baseline:** 논문 곡선과 매우 잘 맞습니다. Fig. 1/3/6/8 전체에서 평균 절대오차는 대체로 **0.05~0.08 dB 수준**입니다.
- **SCI-EGN (Fig. 1):** 논문 EGN 곡선과 전반적으로 잘 맞습니다. 전체 MAE는 약 **0.14 dB**이며, 50-span에서는 세 fiber 모두 약 **0.03 dB 이내**입니다.
- **Full XCI/MCI EGN (Fig. 3/6/8):** 한 가지 quadrature setting (`frequency_order=64`)에서는 MAE가 대략 **0.30~0.46 dB**이지만, **frequency_order 변화에 대한 correction 적분의 수렴이 불안정**합니다.
- 따라서 현재 버전은 **GN 및 SCI-EGN 연구/예측에는 유용**하지만, **Full EGN XCI/MCI를 다른 실험의 정량 sign-off 모델로 사용하기에는 아직 이릅니다.** XCI/MCI 적분 안정화와 독립 SSFM 검증이 필요합니다.

> 주의: 아래 Full-EGN 수치는 `frequency_order=64`, `receiver_points=5`, Sobol `2^16` 샘플, seed=7에서의 비교입니다. 이 setting은 fitting을 하지 않았지만, quadrature-order sensitivity가 확인되므로 결과를 “최종 정확도”로 해석하면 안 됩니다.



## 1. 논문 조건

Carena 2014 Sect. 3 / Fig. 1 조건:
- PM-QPSK, **32 GBaud**
- SSFM simulation은 raised-cosine power spectrum, roll-off = **0.05**
- EGN analytical equations는 paper의 rectangular-spectrum 가정에 따라 계산
- SMF: D=16.7 ps/(nm·km), γ=1.3 1/(W·km), α=0.22 dB/km
- NZDSF: D=3.8, γ=1.5, α=0.22
- LS: D=-1.8, γ=2.2, α=0.22
- span length = **100 km**

Fig. 3 / 6:
- 3 PM-QPSK channels
- center CUT
- spacing = 1.05 × 32 GBd = **33.6 GHz**
- SCI removed

Fig. 8:
- 9 PM-QPSK channels
- center CUT + 4 interferers on each side
- spacing = **33.6 GHz**
- SCI removed



## 2. Paper curve extraction method

논문 PDF의 그래프가 vector path로 보존되어 있어, **OCR 대신 PDF vector 좌표를 직접 추출**했습니다.
x-axis는 1~50 span의 log scale, y-axis는 각 panel의 선형 η[dB(1/W²)] scale로 calibration했습니다.
비교점은 논문의 major span positions인 **1, 2, 5, 10, 20, 50 spans**입니다.

이 방법은 화면에서 수동으로 점을 찍는 것보다 재현성이 높지만, line thickness/plot calibration에 따른 약 **0.05~0.1 dB 정도의 digitization uncertainty**는 고려해야 합니다.


In [ ]:

# Colab에서 실행할 때: final_EGN.py와 논문 PDF를 업로드하세요.
# 이미 같은 작업 디렉터리에 있으면 이 셀은 건너뛰어도 됩니다.
import os, sys, math, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists("final_EGN.py"):
    try:
        from google.colab import files
        print("final_EGN.py를 업로드하세요.")
        files.upload()
    except ImportError:
        pass

spec = importlib.util.spec_from_file_location("final_EGN", "final_EGN.py")
final_EGN = importlib.util.module_from_spec(spec)
sys.modules["final_EGN"] = final_EGN
spec.loader.exec_module(final_EGN)
print("Loaded:", final_EGN.__file__)


In [ ]:
paper = pd.DataFrame([{'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'GN', 'span': 1, 'eta_db': 22.365082226973374, 'x_pdf': 216.9, 'y_pdf': 182.9987168223041}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'GN', 'span': 2, 'eta_db': 26.09010901229179, 'x_pdf': 249.97313186650325, 'y_pdf': 165.64009200272025}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'GN', 'span': 5, 'eta_db': 31.042612855156406, 'x_pdf': 293.6934340667484, 'y_pdf': 142.56142409497116}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'GN', 'span': 10, 'eta_db': 34.52917535222733, 'x_pdf': 326.7665659332516, 'y_pdf': 126.31404285862064}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'GN', 'span': 20, 'eta_db': 38.12439774749046, 'x_pdf': 359.83969779975484, 'y_pdf': 109.56030649669444}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'GN', 'span': 50, 'eta_db': 42.78480601822358, 'x_pdf': 403.56, 'y_pdf': 87.84280395507812}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 1, 'eta_db': 15.55453025499918, 'x_pdf': 216.9, 'y_pdf': 214.73588901170385}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 2, 'eta_db': 22.05614764517045, 'x_pdf': 249.97313186650325, 'y_pdf': 184.4383519735057}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 5, 'eta_db': 28.650180667155457, 'x_pdf': 293.6934340667484, 'y_pdf': 153.71015809105558}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 10, 'eta_db': 32.59774077549275, 'x_pdf': 326.7665659332516, 'y_pdf': 135.31452798620376}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 20, 'eta_db': 36.55500309420901, 'x_pdf': 359.83969779975484, 'y_pdf': 116.873685580986}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 50, 'eta_db': 41.870796367334194, 'x_pdf': 403.56, 'y_pdf': 92.10208892822266}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'EGN', 'span': 1, 'eta_db': 18.129168854492807, 'x_pdf': 216.9, 'y_pdf': 202.73807313806356}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'EGN', 'span': 2, 'eta_db': 22.22751232411071, 'x_pdf': 249.97313186650325, 'y_pdf': 183.6397925696441}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'EGN', 'span': 5, 'eta_db': 28.56261493794949, 'x_pdf': 293.6934340667484, 'y_pdf': 154.11821438915536}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'EGN', 'span': 10, 'eta_db': 32.59784494888583, 'x_pdf': 326.7665659332516, 'y_pdf': 135.31404253819204}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'EGN', 'span': 20, 'eta_db': 36.54623314353547, 'x_pdf': 359.83969779975484, 'y_pdf': 116.91455355112473}, {'fig': 1, 'page': 8, 'fiber': 'SMF', 'curve': 'EGN', 'span': 50, 'eta_db': 41.407246798404806, 'x_pdf': 403.56, 'y_pdf': 94.2622299194336}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 1, 'eta_db': 25.20014949284077, 'x_pdf': 216.9, 'y_pdf': 349.37768859716743}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 2, 'eta_db': 30.08878696148515, 'x_pdf': 249.97313186650325, 'y_pdf': 329.8510737938393}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 5, 'eta_db': 35.84981054049737, 'x_pdf': 293.6934340667484, 'y_pdf': 306.83989961252763}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 10, 'eta_db': 39.82260832114768, 'x_pdf': 326.7665659332516, 'y_pdf': 290.97141019153014}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 20, 'eta_db': 43.692632711969, 'x_pdf': 359.83969779975484, 'y_pdf': 275.51342705333525}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 50, 'eta_db': 48.70777356061812, 'x_pdf': 403.56, 'y_pdf': 255.4815216064453}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 1, 'eta_db': 17.148943361439102, 'x_pdf': 216.9, 'y_pdf': 381.5365062305947}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 2, 'eta_db': 23.76887418968149, 'x_pdf': 249.97313186650325, 'y_pdf': 355.0946110937865}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 5, 'eta_db': 31.67970194592366, 'x_pdf': 293.6934340667484, 'y_pdf': 323.4965047988535}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 10, 'eta_db': 36.717599430593495, 'x_pdf': 326.7665659332516, 'y_pdf': 303.37370284580084}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 20, 'eta_db': 41.33291725245138, 'x_pdf': 359.83969779975484, 'y_pdf': 284.93880480306564}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 50, 'eta_db': 46.664819807453725, 'x_pdf': 403.56, 'y_pdf': 263.64166259765625}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 1, 'eta_db': 21.114164338234136, 'x_pdf': 216.9, 'y_pdf': 365.6982807289962}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 2, 'eta_db': 25.573594291388304, 'x_pdf': 249.97313186650325, 'y_pdf': 347.8860433732547}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 5, 'eta_db': 32.202673451720734, 'x_pdf': 293.6934340667484, 'y_pdf': 321.4076071842697}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 10, 'eta_db': 36.82273733633984, 'x_pdf': 326.7665659332516, 'y_pdf': 302.9537520108483}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 20, 'eta_db': 41.22765671179341, 'x_pdf': 359.83969779975484, 'y_pdf': 285.35924547689376}, {'fig': 1, 'page': 8, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 50, 'eta_db': 46.769835858215416, 'x_pdf': 403.56, 'y_pdf': 263.2221984863281}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'GN', 'span': 1, 'eta_db': 28.70214466565629, 'x_pdf': 216.9, 'y_pdf': 528.1160658406054}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'GN', 'span': 2, 'eta_db': 34.33562622347044, 'x_pdf': 249.97313186650325, 'y_pdf': 505.6239883294926}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'GN', 'span': 5, 'eta_db': 40.86497680108543, 'x_pdf': 293.6934340667484, 'y_pdf': 479.55508976618063}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'GN', 'span': 10, 'eta_db': 45.35956023029175, 'x_pdf': 326.7665659332516, 'y_pdf': 461.6101443834009}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'GN', 'span': 20, 'eta_db': 49.44159894286417, 'x_pdf': 359.83969779975484, 'y_pdf': 445.31231324926176}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'GN', 'span': 50, 'eta_db': 54.563957357399794, 'x_pdf': 403.56, 'y_pdf': 424.8609313964844}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'Simulation', 'span': 1, 'eta_db': 20.31659872265888, 'x_pdf': 216.9, 'y_pdf': 561.59595698559}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'Simulation', 'span': 2, 'eta_db': 26.629974180047046, 'x_pdf': 249.97313186650325, 'y_pdf': 536.3893545165779}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'Simulation', 'span': 5, 'eta_db': 35.508876593445734, 'x_pdf': 293.6934340667484, 'y_pdf': 500.9397024237684}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'Simulation', 'span': 10, 'eta_db': 41.0702241374541, 'x_pdf': 326.7665659332516, 'y_pdf': 478.7356251152047}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'Simulation', 'span': 20, 'eta_db': 46.118894033003066, 'x_pdf': 359.83969779975484, 'y_pdf': 458.5784499379472}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'Simulation', 'span': 50, 'eta_db': 51.78373916339888, 'x_pdf': 403.56, 'y_pdf': 435.9611511230469}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'EGN', 'span': 1, 'eta_db': 24.61444922479395, 'x_pdf': 216.9, 'y_pdf': 544.4364818664941}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'EGN', 'span': 2, 'eta_db': 30.03530119782734, 'x_pdf': 249.97313186650325, 'y_pdf': 522.7933431604459}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'EGN', 'span': 5, 'eta_db': 36.90130205071907, 'x_pdf': 293.6934340667484, 'y_pdf': 495.38034432664335}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'EGN', 'span': 10, 'eta_db': 42.02000779843118, 'x_pdf': 326.7665659332516, 'y_pdf': 474.9435460070636}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'EGN', 'span': 20, 'eta_db': 46.649701703377566, 'x_pdf': 359.83969779975484, 'y_pdf': 456.4591623991434}, {'fig': 1, 'page': 8, 'fiber': 'LS', 'curve': 'EGN', 'span': 50, 'eta_db': 52.20446527507559, 'x_pdf': 403.56, 'y_pdf': 434.2813720703125}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'GN', 'span': 1, 'eta_db': 24.75578701583584, 'x_pdf': 216.84, 'y_pdf': 171.29998621007832}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'GN', 'span': 2, 'eta_db': 27.82042516850064, 'x_pdf': 249.94502495412763, 'y_pdf': 156.99425531343903}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'GN', 'span': 5, 'eta_db': 31.771638722721868, 'x_pdf': 293.70748752293616, 'y_pdf': 138.54999044233432}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'GN', 'span': 10, 'eta_db': 34.79833373646353, 'x_pdf': 326.8125124770638, 'y_pdf': 124.42137811818824}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'GN', 'span': 20, 'eta_db': 37.82897359601136, 'x_pdf': 359.9175374311915, 'y_pdf': 110.27435125381898}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'GN', 'span': 50, 'eta_db': 41.77378205836279, 'x_pdf': 403.68, 'y_pdf': 91.8599853515625}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 1, 'eta_db': 19.884324037825795, 'x_pdf': 216.84, 'y_pdf': 194.0399753914292}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 2, 'eta_db': 24.976827047195485, 'x_pdf': 249.94502495412763, 'y_pdf': 170.26817134369148}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 5, 'eta_db': 29.84619875802488, 'x_pdf': 293.70748752293616, 'y_pdf': 147.5379441975399}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 10, 'eta_db': 33.13696386943122, 'x_pdf': 326.8125124770638, 'y_pdf': 132.17665265749508}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 20, 'eta_db': 36.2793308256632, 'x_pdf': 359.9175374311915, 'y_pdf': 117.50808370580418}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 50, 'eta_db': 40.48843664906428, 'x_pdf': 403.68, 'y_pdf': 97.85997772216795}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'EGN', 'span': 1, 'eta_db': 19.601546120621027, 'x_pdf': 216.84, 'y_pdf': 195.35998270894103}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'EGN', 'span': 2, 'eta_db': 24.706902075981905, 'x_pdf': 249.94502495412763, 'y_pdf': 171.52818110931648}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'EGN', 'span': 5, 'eta_db': 29.654477640385903, 'x_pdf': 293.70748752293616, 'y_pdf': 148.43289837467862}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'EGN', 'span': 10, 'eta_db': 32.97214928531909, 'x_pdf': 326.8125124770638, 'y_pdf': 132.94600713613048}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'EGN', 'span': 20, 'eta_db': 36.08090246646662, 'x_pdf': 359.9175374311915, 'y_pdf': 118.43434728653382}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'EGN', 'span': 50, 'eta_db': 40.11568482962039, 'x_pdf': 403.68, 'y_pdf': 99.59998321533205}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'XPM', 'span': 1, 'eta_db': 17.763496248848185, 'x_pdf': 216.84, 'y_pdf': 203.93999951037668}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'XPM', 'span': 2, 'eta_db': 23.423158599782145, 'x_pdf': 249.94502495412763, 'y_pdf': 177.52069565621696}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'XPM', 'span': 5, 'eta_db': 28.458386505395797, 'x_pdf': 293.70748752293616, 'y_pdf': 154.01625179281243}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'XPM', 'span': 10, 'eta_db': 31.851610288116284, 'x_pdf': 326.8125124770638, 'y_pdf': 138.1766831750732}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'XPM', 'span': 20, 'eta_db': 34.98741553263095, 'x_pdf': 359.9175374311915, 'y_pdf': 123.53874429367876}, {'fig': 3, 'page': 13, 'fiber': 'SMF', 'curve': 'XPM', 'span': 50, 'eta_db': 39.0102823187167, 'x_pdf': 403.68, 'y_pdf': 104.76000213623048}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 1, 'eta_db': 30.97165826053958, 'x_pdf': 218.34, 'y_pdf': 338.3379024554488}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 2, 'eta_db': 33.99727586840734, 'x_pdf': 250.88158040609656, 'y_pdf': 324.2566781084322}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 5, 'eta_db': 38.06168273440133, 'x_pdf': 293.8992097969517, 'y_pdf': 305.3409285540962}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 10, 'eta_db': 41.04958451104288, 'x_pdf': 326.4407902030483, 'y_pdf': 291.4352336856064}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 20, 'eta_db': 44.04271571032052, 'x_pdf': 358.9823706091448, 'y_pdf': 277.50520108416833}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 50, 'eta_db': 48.10521537672567, 'x_pdf': 402.0, 'y_pdf': 258.59832763671875}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 1, 'eta_db': 27.078246372496707, 'x_pdf': 218.34, 'y_pdf': 356.4578413824003}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 2, 'eta_db': 31.10380052341388, 'x_pdf': 250.88158040609656, 'y_pdf': 337.7229123640318}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 5, 'eta_db': 35.62473485311602, 'x_pdf': 293.8992097969517, 'y_pdf': 316.68248399359805}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 10, 'eta_db': 38.79282078419056, 'x_pdf': 326.4407902030483, 'y_pdf': 301.9382120703771}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 20, 'eta_db': 41.95816297650946, 'x_pdf': 358.9823706091448, 'y_pdf': 287.20670950732494}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 50, 'eta_db': 46.10689760740089, 'x_pdf': 402.0, 'y_pdf': 267.89849853515625}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 1, 'eta_db': 26.433647603392647, 'x_pdf': 218.34, 'y_pdf': 359.4578040538106}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 2, 'eta_db': 30.201122494765784, 'x_pdf': 250.88158040609656, 'y_pdf': 341.92397590936}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 5, 'eta_db': 34.890241459087086, 'x_pdf': 293.8992097969517, 'y_pdf': 320.1008162494087}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 10, 'eta_db': 38.05854054810442, 'x_pdf': 326.4407902030483, 'y_pdf': 305.35555228912204}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 20, 'eta_db': 41.14197260145953, 'x_pdf': 358.9823706091448, 'y_pdf': 291.00525951280736}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 50, 'eta_db': 45.29478511638174, 'x_pdf': 402.0, 'y_pdf': 271.6780700683594}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 1, 'eta_db': 24.08721711874428, 'x_pdf': 218.34, 'y_pdf': 370.3780915293641}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 2, 'eta_db': 28.30041985649465, 'x_pdf': 250.88158040609656, 'y_pdf': 350.7698459878739}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 5, 'eta_db': 33.35827478706943, 'x_pdf': 293.8992097969517, 'y_pdf': 327.23058914097885}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 10, 'eta_db': 36.69989689605893, 'x_pdf': 326.4407902030483, 'y_pdf': 311.6786798457417}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 20, 'eta_db': 39.86954491316678, 'x_pdf': 358.9823706091448, 'y_pdf': 296.9271379741218}, {'fig': 3, 'page': 13, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 50, 'eta_db': 44.10858639535547, 'x_pdf': 402.0, 'y_pdf': 277.1986389160156}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'GN', 'span': 1, 'eta_db': 36.06182690526829, 'x_pdf': 218.34, 'y_pdf': 510.1782575828814}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'GN', 'span': 2, 'eta_db': 39.26965291191831, 'x_pdf': 250.88158040609656, 'y_pdf': 495.2490353479322}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'GN', 'span': 5, 'eta_db': 43.43225990562964, 'x_pdf': 293.8992097969517, 'y_pdf': 475.8762623991997}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'GN', 'span': 10, 'eta_db': 46.3324005223862, 'x_pdf': 326.4407902030483, 'y_pdf': 462.3790079688146}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'GN', 'span': 20, 'eta_db': 49.40763377166084, 'x_pdf': 358.9823706091448, 'y_pdf': 448.06687242669045}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'GN', 'span': 50, 'eta_db': 53.55638688451803, 'x_pdf': 402.0, 'y_pdf': 428.7585754394531}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'Simulation', 'span': 1, 'eta_db': 32.52938772954245, 'x_pdf': 218.34, 'y_pdf': 526.6182295067094}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'Simulation', 'span': 2, 'eta_db': 35.73902427695191, 'x_pdf': 250.88158040609656, 'y_pdf': 511.6805810150658}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'Simulation', 'span': 5, 'eta_db': 39.98040844041043, 'x_pdf': 293.8992097969517, 'y_pdf': 491.9411791183298}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'Simulation', 'span': 10, 'eta_db': 43.058407304187895, 'x_pdf': 326.4407902030483, 'y_pdf': 477.6161724063096}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'Simulation', 'span': 20, 'eta_db': 46.141839357543006, 'x_pdf': 358.9823706091448, 'y_pdf': 463.26587962999486}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'Simulation', 'span': 50, 'eta_db': 50.384958723762495, 'x_pdf': 402.0, 'y_pdf': 443.5184020996094}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'EGN', 'span': 1, 'eta_db': 31.52379017821725, 'x_pdf': 218.34, 'y_pdf': 531.2982805105769}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'EGN', 'span': 2, 'eta_db': 34.746039397006555, 'x_pdf': 250.88158040609656, 'y_pdf': 516.3019326463315}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'EGN', 'span': 5, 'eta_db': 38.891457523080106, 'x_pdf': 293.8992097969517, 'y_pdf': 497.0091566875852}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'EGN', 'span': 10, 'eta_db': 41.61396590129413, 'x_pdf': 326.4407902030483, 'y_pdf': 484.3386026953771}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'EGN', 'span': 20, 'eta_db': 44.68900124231576, 'x_pdf': 358.9823706091448, 'y_pdf': 470.0273882182624}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'EGN', 'span': 50, 'eta_db': 49.10865642709901, 'x_pdf': 402.0, 'y_pdf': 449.4583129882813}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'XPM', 'span': 1, 'eta_db': 29.08723981441461, 'x_pdf': 218.34, 'y_pdf': 542.6379859037144}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'XPM', 'span': 2, 'eta_db': 32.39236732276228, 'x_pdf': 250.88158040609656, 'y_pdf': 527.2559224798644}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'XPM', 'span': 5, 'eta_db': 37.546219345246655, 'x_pdf': 293.8992097969517, 'y_pdf': 503.2698951672221}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'XPM', 'span': 10, 'eta_db': 40.80183653552408, 'x_pdf': 326.4407902030483, 'y_pdf': 488.1182527636709}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'XPM', 'span': 20, 'eta_db': 44.147592914969664, 'x_pdf': 358.9823706091448, 'y_pdf': 472.5471025737312}, {'fig': 3, 'page': 13, 'fiber': 'LS', 'curve': 'XPM', 'span': 50, 'eta_db': 48.74763885484933, 'x_pdf': 402.0, 'y_pdf': 451.1384887695313}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'GN', 'span': 1, 'eta_db': 24.759659822999375, 'x_pdf': 216.9, 'y_pdf': 171.2999852248229}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'GN', 'span': 2, 'eta_db': 27.744628531218343, 'x_pdf': 249.95186980808697, 'y_pdf': 157.39003104452254}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'GN', 'span': 5, 'eta_db': 31.770185246896105, 'x_pdf': 293.64406509595653, 'y_pdf': 138.63093674946415}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'GN', 'span': 10, 'eta_db': 34.79521989559194, 'x_pdf': 326.6959349040435, 'y_pdf': 124.53427528654156}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'GN', 'span': 20, 'eta_db': 37.853808598120445, 'x_pdf': 359.74780471213046, 'y_pdf': 110.28125193275872}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'GN', 'span': 50, 'eta_db': 41.76824323793271, 'x_pdf': 403.44, 'y_pdf': 92.03998651123358}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 1, 'eta_db': 19.879833978935537, 'x_pdf': 216.9, 'y_pdf': 194.0399736581604}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 2, 'eta_db': 24.95320627151952, 'x_pdf': 249.95186980808697, 'y_pdf': 170.39805877471906}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 5, 'eta_db': 29.855403385272524, 'x_pdf': 293.64406509595653, 'y_pdf': 147.55382022463004}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 10, 'eta_db': 33.0579979448725, 'x_pdf': 326.6959349040435, 'y_pdf': 132.62972957689416}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 20, 'eta_db': 36.2892970220901, 'x_pdf': 359.74780471213046, 'y_pdf': 117.57187587706014}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 50, 'eta_db': 40.4678162144994, 'x_pdf': 403.44, 'y_pdf': 98.0999764404328}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'EGN', 'span': 1, 'eta_db': 19.78969579324616, 'x_pdf': 216.9, 'y_pdf': 194.4600176034729}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'EGN', 'span': 2, 'eta_db': 24.86306808583014, 'x_pdf': 249.95186980808697, 'y_pdf': 170.81810272003156}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'EGN', 'span': 5, 'eta_db': 29.76893565641564, 'x_pdf': 293.64406509595653, 'y_pdf': 147.95675984110315}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'EGN', 'span': 10, 'eta_db': 33.05701461209543, 'x_pdf': 326.6959349040435, 'y_pdf': 132.63431190763532}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'EGN', 'span': 20, 'eta_db': 36.21113509413786, 'x_pdf': 359.74780471213046, 'y_pdf': 117.93611046131758}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'EGN', 'span': 50, 'eta_db': 40.28754802916961, 'x_pdf': 403.44, 'y_pdf': 98.9400261840696}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'XPM', 'span': 1, 'eta_db': 17.781115343515864, 'x_pdf': 216.9, 'y_pdf': 203.8200024992161}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'XPM', 'span': 2, 'eta_db': 23.38344289085461, 'x_pdf': 249.95186980808697, 'y_pdf': 177.71315612861753}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'XPM', 'span': 5, 'eta_db': 28.467982682895723, 'x_pdf': 293.64406509595653, 'y_pdf': 154.01920069770594}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'XPM', 'span': 10, 'eta_db': 31.83383934257648, 'x_pdf': 326.6959349040435, 'y_pdf': 138.3343086635936}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'XPM', 'span': 20, 'eta_db': 34.98796153504627, 'x_pdf': 359.74780471213046, 'y_pdf': 123.63609924668438}, {'fig': 6, 'page': 17, 'fiber': 'SMF', 'curve': 'XPM', 'span': 50, 'eta_db': 39.06437599280628, 'x_pdf': 403.44, 'y_pdf': 104.64000787352272}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 1, 'eta_db': 30.98283238088338, 'x_pdf': 216.9, 'y_pdf': 335.16000110508344}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 2, 'eta_db': 33.9544698377033, 'x_pdf': 249.95186980808697, 'y_pdf': 321.3121705563026}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 5, 'eta_db': 38.08348630294605, 'x_pdf': 293.64406509595653, 'y_pdf': 302.0709538282714}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 10, 'eta_db': 41.01838922270658, 'x_pdf': 326.6959349040435, 'y_pdf': 288.3943062221873}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 20, 'eta_db': 44.0769794067007, 'x_pdf': 359.74780471213046, 'y_pdf': 274.1412759647747}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 50, 'eta_db': 48.08154252343844, 'x_pdf': 403.44, 'y_pdf': 255.48001184077685}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 1, 'eta_db': 27.05579656943916, 'x_pdf': 216.9, 'y_pdf': 353.4599879864135}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 2, 'eta_db': 31.084740567244783, 'x_pdf': 249.95186980808697, 'y_pdf': 334.6851089566393}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 5, 'eta_db': 35.55068078350884, 'x_pdf': 293.64406509595653, 'y_pdf': 313.8738275488488}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 10, 'eta_db': 38.75230179232386, 'x_pdf': 326.6959349040435, 'y_pdf': 298.9542736477708}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 20, 'eta_db': 41.90641927017242, 'x_pdf': 359.74780471213046, 'y_pdf': 284.2560862009965}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 50, 'eta_db': 46.07296197587952, 'x_pdf': 403.44, 'y_pdf': 264.8399971924014}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 1, 'eta_db': 27.23605322750816, 'x_pdf': 216.9, 'y_pdf': 352.61999195981195}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 2, 'eta_db': 30.90448384237555, 'x_pdf': 249.95186980808697, 'y_pdf': 335.5251052945299}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 5, 'eta_db': 35.55068078350884, 'x_pdf': 293.64406509595653, 'y_pdf': 313.8738275488488}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 10, 'eta_db': 38.66217023799628, 'x_pdf': 326.6959349040435, 'y_pdf': 299.3742866909373}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 20, 'eta_db': 41.81088596793024, 'x_pdf': 359.74780471213046, 'y_pdf': 284.7012713894451}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 50, 'eta_db': 45.892705277217615, 'x_pdf': 403.44, 'y_pdf': 265.6799934081659}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 1, 'eta_db': 24.00429420927808, 'x_pdf': 216.9, 'y_pdf': 367.6799889847641}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 2, 'eta_db': 28.29278984948702, 'x_pdf': 249.95186980808697, 'y_pdf': 347.6955993013905}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 5, 'eta_db': 33.37471224152144, 'x_pdf': 293.64406509595653, 'y_pdf': 324.0138409545101}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 10, 'eta_db': 36.65358314161418, 'x_pdf': 326.6959349040435, 'y_pdf': 308.73430256007794}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 20, 'eta_db': 39.89243659931404, 'x_pdf': 359.74780471213046, 'y_pdf': 293.6412454471966}, {'fig': 6, 'page': 17, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 50, 'eta_db': 44.154506332992185, 'x_pdf': 403.44, 'y_pdf': 273.7800004882564}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'GN', 'span': 1, 'eta_db': 36.06483791273655, 'x_pdf': 216.9, 'y_pdf': 504.2999850024732}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'GN', 'span': 2, 'eta_db': 39.308180182302486, 'x_pdf': 249.95186980808697, 'y_pdf': 489.19249671083503}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'GN', 'span': 5, 'eta_db': 43.422492468669205, 'x_pdf': 293.64406509595653, 'y_pdf': 470.0280300809389}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'GN', 'span': 10, 'eta_db': 46.30702938825459, 'x_pdf': 326.6959349040435, 'y_pdf': 456.59185710951016}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'GN', 'span': 20, 'eta_db': 49.35034567129544, 'x_pdf': 359.74780471213046, 'y_pdf': 442.41608986310587}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'GN', 'span': 50, 'eta_db': 53.51867736057349, 'x_pdf': 403.44, 'y_pdf': 423.00000085444873}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'Simulation', 'span': 1, 'eta_db': 32.49677089610692, 'x_pdf': 216.9, 'y_pdf': 520.920041165934}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'Simulation', 'span': 2, 'eta_db': 35.72828999728142, 'x_pdf': 249.95186980808697, 'y_pdf': 505.8676251926632}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'Simulation', 'span': 5, 'eta_db': 40.02554478317984, 'x_pdf': 293.64406509595653, 'y_pdf': 485.85101239994833}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'Simulation', 'span': 10, 'eta_db': 43.05188439761791, 'x_pdf': 326.6959349040435, 'y_pdf': 471.7543224758958}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'Simulation', 'span': 20, 'eta_db': 46.201051048258606, 'x_pdf': 359.74780471213046, 'y_pdf': 457.08550421721145}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'Simulation', 'span': 50, 'eta_db': 50.37568784445547, 'x_pdf': 403.44, 'y_pdf': 437.6400460205264}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'EGN', 'span': 1, 'eta_db': 32.934735469888565, 'x_pdf': 216.9, 'y_pdf': 518.8800021812591}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'EGN', 'span': 2, 'eta_db': 36.07563110912712, 'x_pdf': 249.95186980808697, 'y_pdf': 504.2497102936859}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'EGN', 'span': 5, 'eta_db': 40.20221970834758, 'x_pdf': 293.64406509595653, 'y_pdf': 485.028060598517}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'EGN', 'span': 10, 'eta_db': 43.05075474865693, 'x_pdf': 326.6959349040435, 'y_pdf': 471.75958438075605}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'EGN', 'span': 20, 'eta_db': 46.11179447910322, 'x_pdf': 359.74780471213046, 'y_pdf': 457.5012613163372}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'EGN', 'span': 50, 'eta_db': 50.375694396117176, 'x_pdf': 403.44, 'y_pdf': 437.64001550288623}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'XPM', 'span': 1, 'eta_db': 29.096177117594152, 'x_pdf': 216.9, 'y_pdf': 536.7600069862465}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'XPM', 'span': 2, 'eta_db': 32.32928229286317, 'x_pdf': 249.95186980808697, 'y_pdf': 521.7002030798434}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'XPM', 'span': 5, 'eta_db': 37.59103086726904, 'x_pdf': 293.64406509595653, 'y_pdf': 497.1909782202608}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'XPM', 'span': 10, 'eta_db': 40.78481759340454, 'x_pdf': 326.6959349040435, 'y_pdf': 482.3143196499217}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'XPM', 'span': 20, 'eta_db': 44.10865006496611, 'x_pdf': 359.74780471213046, 'y_pdf': 466.8319079973879}, {'fig': 6, 'page': 17, 'fiber': 'LS', 'curve': 'XPM', 'span': 50, 'eta_db': 48.71403915516119, 'x_pdf': 403.44, 'y_pdf': 445.3800056152592}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'GN', 'span': 1, 'eta_db': 27.72746618523028, 'x_pdf': 216.66, 'y_pdf': 173.4000244140625}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'GN', 'span': 2, 'eta_db': 30.805869797783785, 'x_pdf': 249.82881112937645, 'y_pdf': 156.42940097877755}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'GN', 'span': 5, 'eta_db': 34.78854160656258, 'x_pdf': 293.6755944353118, 'y_pdf': 134.4737278313418}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'GN', 'span': 10, 'eta_db': 37.74389947870245, 'x_pdf': 326.84440556468826, 'y_pdf': 118.18143095380913}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'GN', 'span': 20, 'eta_db': 40.79186349740443, 'x_pdf': 360.0132166940647, 'y_pdf': 101.37861491150888}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'GN', 'span': 50, 'eta_db': 44.760553096759615, 'x_pdf': 403.86, 'y_pdf': 79.5000228881836}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 1, 'eta_db': 22.11145237419052, 'x_pdf': 216.66, 'y_pdf': 204.3599853515625}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 2, 'eta_db': 27.304525988220146, 'x_pdf': 249.82881112937645, 'y_pdf': 175.73160913214}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 5, 'eta_db': 32.5181870785712, 'x_pdf': 293.6755944353118, 'y_pdf': 146.9897382732527}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 10, 'eta_db': 35.87189948816197, 'x_pdf': 326.84440556468826, 'y_pdf': 128.5013925016607}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 20, 'eta_db': 39.158635006568986, 'x_pdf': 360.0132166940647, 'y_pdf': 110.38227693578648}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'Simulation', 'span': 50, 'eta_db': 43.35655517976701, 'x_pdf': 403.86, 'y_pdf': 87.23998260498047}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'EGN', 'span': 1, 'eta_db': 22.416196365588945, 'x_pdf': 216.66, 'y_pdf': 202.67999267578125}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'EGN', 'span': 2, 'eta_db': 27.303152079221157, 'x_pdf': 249.82881112937645, 'y_pdf': 175.73918321766962}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'EGN', 'span': 5, 'eta_db': 32.441726836979846, 'x_pdf': 293.6755944353118, 'y_pdf': 147.41124829309751}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'EGN', 'span': 10, 'eta_db': 35.87189944600323, 'x_pdf': 326.84440556468826, 'y_pdf': 128.50139273407336}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'EGN', 'span': 20, 'eta_db': 39.07223411862133, 'x_pdf': 360.0132166940647, 'y_pdf': 110.85858775086436}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'EGN', 'span': 50, 'eta_db': 43.12799441833452, 'x_pdf': 403.86, 'y_pdf': 88.49999237060547}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'XPM', 'span': 1, 'eta_db': 20.620371301363416, 'x_pdf': 216.66, 'y_pdf': 212.58001708984372}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'XPM', 'span': 2, 'eta_db': 26.064944909520523, 'x_pdf': 249.82881112937645, 'y_pdf': 182.56517170279523}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'XPM', 'span': 5, 'eta_db': 31.505719922000445, 'x_pdf': 293.6755944353118, 'y_pdf': 152.57126721399595}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'XPM', 'span': 10, 'eta_db': 34.935889805298906, 'x_pdf': 326.84440556468826, 'y_pdf': 133.6614266813482}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'XPM', 'span': 20, 'eta_db': 38.20771465632532, 'x_pdf': 360.0132166940647, 'y_pdf': 115.62451064260976}, {'fig': 8, 'page': 20, 'fiber': 'SMF', 'curve': 'XPM', 'span': 50, 'eta_db': 42.26817211726289, 'x_pdf': 403.86, 'y_pdf': 93.24002075195312}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 1, 'eta_db': 34.75715700172826, 'x_pdf': 217.92, 'y_pdf': 340.67999190802925}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 2, 'eta_db': 37.73141805112227, 'x_pdf': 250.6316768734267, 'y_pdf': 326.9626999482241}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 5, 'eta_db': 41.76616209545425, 'x_pdf': 293.87416156328663, 'y_pdf': 308.354460415765}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 10, 'eta_db': 44.80197959520284, 'x_pdf': 326.58583843671335, 'y_pdf': 294.3532701069245}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 20, 'eta_db': 47.830659977553466, 'x_pdf': 359.29751531014006, 'y_pdf': 280.3849961835234}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'GN', 'span': 50, 'eta_db': 51.78664697943205, 'x_pdf': 402.54, 'y_pdf': 262.1399841308594}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 1, 'eta_db': 30.607110513470783, 'x_pdf': 217.92, 'y_pdf': 359.8200063118728}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 2, 'eta_db': 34.52664946890872, 'x_pdf': 250.6316768734267, 'y_pdf': 341.74309264939296}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 5, 'eta_db': 39.372404822054015, 'x_pdf': 293.87416156328663, 'y_pdf': 319.3944689606869}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 10, 'eta_db': 42.68141816916749, 'x_pdf': 326.58583843671335, 'y_pdf': 304.1332994037995}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 20, 'eta_db': 45.82014639110426, 'x_pdf': 359.29751531014006, 'y_pdf': 289.6574848442272}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'Simulation', 'span': 50, 'eta_db': 49.93928801775599, 'x_pdf': 402.54, 'y_pdf': 270.6600036621094}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 1, 'eta_db': 31.166524431798504, 'x_pdf': 217.92, 'y_pdf': 357.2399893205453}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 2, 'eta_db': 34.70702036115967, 'x_pdf': 250.6316768734267, 'y_pdf': 340.9112220943316}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 5, 'eta_db': 39.37490760554372, 'x_pdf': 293.87416156328663, 'y_pdf': 319.3829261232324}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 10, 'eta_db': 42.68555776974628, 'x_pdf': 326.58583843671335, 'y_pdf': 304.11420756593014}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 20, 'eta_db': 45.90524695005726, 'x_pdf': 359.29751531014006, 'y_pdf': 289.2650010663359}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'EGN', 'span': 50, 'eta_db': 49.84822494096582, 'x_pdf': 402.54, 'y_pdf': 271.0799865722656}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 1, 'eta_db': 27.484822172503147, 'x_pdf': 217.92, 'y_pdf': 374.2200001404155}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 2, 'eta_db': 31.67187442621252, 'x_pdf': 250.6316768734267, 'y_pdf': 354.90931514630785}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 5, 'eta_db': 37.08487152237792, 'x_pdf': 293.87416156328663, 'y_pdf': 329.944572538793}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 10, 'eta_db': 40.56138021299135, 'x_pdf': 326.58583843671335, 'y_pdf': 313.9109144576839}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 20, 'eta_db': 43.78469170772528, 'x_pdf': 359.29751531014006, 'y_pdf': 299.045001843971}, {'fig': 8, 'page': 20, 'fiber': 'NZDSF', 'curve': 'XPM', 'span': 50, 'eta_db': 47.9097962855052, 'x_pdf': 402.54, 'y_pdf': 280.02001953125}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'GN', 'span': 1, 'eta_db': 40.66782483069133, 'x_pdf': 217.92, 'y_pdf': 505.6799918808516}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'GN', 'span': 2, 'eta_db': 43.74591182661905, 'x_pdf': 250.6316768734267, 'y_pdf': 491.4838546556329}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'GN', 'span': 5, 'eta_db': 47.778405839789365, 'x_pdf': 293.87416156328663, 'y_pdf': 472.8859922668914}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'GN', 'span': 10, 'eta_db': 50.72927255996271, 'x_pdf': 326.58583843671335, 'y_pdf': 459.276594953452}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'GN', 'span': 20, 'eta_db': 53.760299762721935, 'x_pdf': 359.29751531014006, 'y_pdf': 445.2974974943264}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'GN', 'span': 50, 'eta_db': 57.78837787929247, 'x_pdf': 402.54, 'y_pdf': 426.7200012207031}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'Simulation', 'span': 1, 'eta_db': 37.2723406372029, 'x_pdf': 217.92, 'y_pdf': 521.3399649812202}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'Simulation', 'span': 2, 'eta_db': 40.52510936444962, 'x_pdf': 250.6316768734267, 'y_pdf': 506.3381956111584}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'Simulation', 'span': 5, 'eta_db': 45.11395550251396, 'x_pdf': 293.87416156328663, 'y_pdf': 485.1744372224056}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'Simulation', 'span': 10, 'eta_db': 48.33190577283727, 'x_pdf': 326.58583843671335, 'y_pdf': 470.3332505756745}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'Simulation', 'span': 20, 'eta_db': 51.45165541568142, 'x_pdf': 359.29751531014006, 'y_pdf': 455.9449652228773}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'Simulation', 'span': 50, 'eta_db': 55.58977562142576, 'x_pdf': 402.54, 'y_pdf': 436.8599548339844}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'EGN', 'span': 1, 'eta_db': 38.00087170992622, 'x_pdf': 217.92, 'y_pdf': 517.9799796738203}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'EGN', 'span': 2, 'eta_db': 41.0824956964903, 'x_pdf': 250.6316768734267, 'y_pdf': 503.7675298477868}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'EGN', 'span': 5, 'eta_db': 45.38715134987885, 'x_pdf': 293.87416156328663, 'y_pdf': 483.9144579743588}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'EGN', 'span': 10, 'eta_db': 48.5176483198992, 'x_pdf': 326.58583843671335, 'y_pdf': 469.47660594862487}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'EGN', 'span': 20, 'eta_db': 51.63464003641732, 'x_pdf': 359.29751531014006, 'y_pdf': 455.1010401520433}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'EGN', 'span': 50, 'eta_db': 55.589769004431375, 'x_pdf': 402.54, 'y_pdf': 436.8599853515625}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'XPM', 'span': 1, 'eta_db': 33.12229410180363, 'x_pdf': 217.92, 'y_pdf': 540.4799796024816}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'XPM', 'span': 2, 'eta_db': 36.48444026837625, 'x_pdf': 250.6316768734267, 'y_pdf': 524.9737614822487}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'XPM', 'span': 5, 'eta_db': 41.710461056616246, 'x_pdf': 293.87416156328663, 'y_pdf': 500.87135360688586}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'XPM', 'span': 10, 'eta_db': 45.200734772758125, 'x_pdf': 326.58583843671335, 'y_pdf': 484.7742112280395}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'XPM', 'span': 20, 'eta_db': 48.51149364685366, 'x_pdf': 359.29751531014006, 'y_pdf': 469.5049913007109}, {'fig': 8, 'page': 20, 'fiber': 'LS', 'curve': 'XPM', 'span': 50, 'eta_db': 52.72766748512712, 'x_pdf': 402.54, 'y_pdf': 450.05999755859375}])
code = pd.DataFrame([{'fig': 1, 'fiber': 'SMF', 'span': 1, 'curve': 'Code GN', 'eta_db': 22.34173617302435}, {'fig': 1, 'fiber': 'SMF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 18.06048445251217}, {'fig': 1, 'fiber': 'SMF', 'span': 2, 'curve': 'Code GN', 'eta_db': 26.061057666963944}, {'fig': 1, 'fiber': 'SMF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 23.07316553655677}, {'fig': 1, 'fiber': 'SMF', 'span': 5, 'curve': 'Code GN', 'eta_db': 30.92559086460382}, {'fig': 1, 'fiber': 'SMF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 28.777249800386944}, {'fig': 1, 'fiber': 'SMF', 'span': 10, 'curve': 'Code GN', 'eta_db': 34.539293239345675}, {'fig': 1, 'fiber': 'SMF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 32.76336401592337}, {'fig': 1, 'fiber': 'SMF', 'span': 20, 'curve': 'Code GN', 'eta_db': 38.093053118806495}, {'fig': 1, 'fiber': 'SMF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 36.57723189817806}, {'fig': 1, 'fiber': 'SMF', 'span': 50, 'curve': 'Code GN', 'eta_db': 42.70889106002152}, {'fig': 1, 'fiber': 'SMF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 41.43688173537309}, {'fig': 1, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code GN', 'eta_db': 25.11130956024032}, {'fig': 1, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 21.04390388279973}, {'fig': 1, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code GN', 'eta_db': 29.927357018513405}, {'fig': 1, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 25.85021016169907}, {'fig': 1, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code GN', 'eta_db': 35.73486248847326}, {'fig': 1, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 32.378834775149045}, {'fig': 1, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code GN', 'eta_db': 39.82240641961315}, {'fig': 1, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 37.00582764847546}, {'fig': 1, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code GN', 'eta_db': 43.72389880618843}, {'fig': 1, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 41.35737065536748}, {'fig': 1, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code GN', 'eta_db': 48.67417380062872}, {'fig': 1, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 46.76491891614791}, {'fig': 1, 'fiber': 'LS', 'span': 1, 'curve': 'Code GN', 'eta_db': 28.61094303700018}, {'fig': 1, 'fiber': 'LS', 'span': 1, 'curve': 'Code EGN', 'eta_db': 24.60816768799028}, {'fig': 1, 'fiber': 'LS', 'span': 2, 'curve': 'Code GN', 'eta_db': 34.2352747258212}, {'fig': 1, 'fiber': 'LS', 'span': 2, 'curve': 'Code EGN', 'eta_db': 30.098720877817904}, {'fig': 1, 'fiber': 'LS', 'span': 5, 'curve': 'Code GN', 'eta_db': 40.772011179653774}, {'fig': 1, 'fiber': 'LS', 'span': 5, 'curve': 'Code EGN', 'eta_db': 36.93861942391155}, {'fig': 1, 'fiber': 'LS', 'span': 10, 'curve': 'Code GN', 'eta_db': 45.1880691459045}, {'fig': 1, 'fiber': 'LS', 'span': 10, 'curve': 'Code EGN', 'eta_db': 41.88406981981301}, {'fig': 1, 'fiber': 'LS', 'span': 20, 'curve': 'Code GN', 'eta_db': 49.31361156731356}, {'fig': 1, 'fiber': 'LS', 'span': 20, 'curve': 'Code EGN', 'eta_db': 46.51695286768003}, {'fig': 1, 'fiber': 'LS', 'span': 50, 'curve': 'Code GN', 'eta_db': 54.465178235593456}, {'fig': 1, 'fiber': 'LS', 'span': 50, 'curve': 'Code EGN', 'eta_db': 52.22055291458672}, {'fig': 3, 'fiber': 'SMF', 'span': 1, 'curve': 'Code GN', 'eta_db': 24.68083295778112}, {'fig': 3, 'fiber': 'SMF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 19.535012875902005}, {'fig': 6, 'fiber': 'SMF', 'span': 1, 'curve': 'Code GN', 'eta_db': 24.68083295778112}, {'fig': 6, 'fiber': 'SMF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 19.73674221067618}, {'fig': 3, 'fiber': 'SMF', 'span': 2, 'curve': 'Code GN', 'eta_db': 27.717708391525647}, {'fig': 3, 'fiber': 'SMF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 24.73497030668344}, {'fig': 6, 'fiber': 'SMF', 'span': 2, 'curve': 'Code GN', 'eta_db': 27.717708391525647}, {'fig': 6, 'fiber': 'SMF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 24.87504358920184}, {'fig': 3, 'fiber': 'SMF', 'span': 5, 'curve': 'Code GN', 'eta_db': 31.746941322963032}, {'fig': 3, 'fiber': 'SMF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 29.84229745168996}, {'fig': 6, 'fiber': 'SMF', 'span': 5, 'curve': 'Code GN', 'eta_db': 31.746941322963032}, {'fig': 6, 'fiber': 'SMF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 29.960303252178093}, {'fig': 3, 'fiber': 'SMF', 'span': 10, 'curve': 'Code GN', 'eta_db': 34.78299402856548}, {'fig': 3, 'fiber': 'SMF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 33.34277766082415}, {'fig': 6, 'fiber': 'SMF', 'span': 10, 'curve': 'Code GN', 'eta_db': 34.78299402856548}, {'fig': 6, 'fiber': 'SMF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 33.44776461316552}, {'fig': 3, 'fiber': 'SMF', 'span': 20, 'curve': 'Code GN', 'eta_db': 37.815877174973565}, {'fig': 3, 'fiber': 'SMF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 37.16495513385082}, {'fig': 6, 'fiber': 'SMF', 'span': 20, 'curve': 'Code GN', 'eta_db': 37.815877174973565}, {'fig': 6, 'fiber': 'SMF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 37.23420848749746}, {'fig': 3, 'fiber': 'SMF', 'span': 50, 'curve': 'Code GN', 'eta_db': 41.88643494503509}, {'fig': 3, 'fiber': 'SMF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 41.60660032700102}, {'fig': 6, 'fiber': 'SMF', 'span': 50, 'curve': 'Code GN', 'eta_db': 41.88643494503509}, {'fig': 6, 'fiber': 'SMF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 41.66153025997544}, {'fig': 3, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code GN', 'eta_db': 30.95714219440612}, {'fig': 3, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 26.418286003470687}, {'fig': 6, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code GN', 'eta_db': 30.95714219440612}, {'fig': 6, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 27.179873723780595}, {'fig': 3, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code GN', 'eta_db': 33.98422141457496}, {'fig': 3, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 30.390906574434872}, {'fig': 6, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code GN', 'eta_db': 33.98422141457496}, {'fig': 6, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 31.061021321964443}, {'fig': 3, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code GN', 'eta_db': 37.98988237711307}, {'fig': 3, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 34.88620566460042}, {'fig': 6, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code GN', 'eta_db': 37.98988237711307}, {'fig': 6, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 35.522469122678594}, {'fig': 3, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code GN', 'eta_db': 41.02359870562746}, {'fig': 3, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 38.08358176171653}, {'fig': 6, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code GN', 'eta_db': 41.02359870562746}, {'fig': 6, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 38.7039248574977}, {'fig': 3, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code GN', 'eta_db': 44.08983122979705}, {'fig': 3, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 41.49285206116495}, {'fig': 6, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code GN', 'eta_db': 44.08983122979705}, {'fig': 6, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 42.07756539328932}, {'fig': 3, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code GN', 'eta_db': 48.09636945009973}, {'fig': 3, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 46.13603433575677}, {'fig': 6, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code GN', 'eta_db': 48.09636945009973}, {'fig': 6, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 46.652947885210914}, {'fig': 3, 'fiber': 'LS', 'span': 1, 'curve': 'Code GN', 'eta_db': 36.04098850935108}, {'fig': 3, 'fiber': 'LS', 'span': 1, 'curve': 'Code EGN', 'eta_db': 31.501427209469814}, {'fig': 6, 'fiber': 'LS', 'span': 1, 'curve': 'Code GN', 'eta_db': 36.04098850935108}, {'fig': 6, 'fiber': 'LS', 'span': 1, 'curve': 'Code EGN', 'eta_db': 32.85945877962699}, {'fig': 3, 'fiber': 'LS', 'span': 2, 'curve': 'Code GN', 'eta_db': 39.24075318415365}, {'fig': 3, 'fiber': 'LS', 'span': 2, 'curve': 'Code EGN', 'eta_db': 34.70247243997579}, {'fig': 6, 'fiber': 'LS', 'span': 2, 'curve': 'Code GN', 'eta_db': 39.24075318415365}, {'fig': 6, 'fiber': 'LS', 'span': 2, 'curve': 'Code EGN', 'eta_db': 35.9995283113233}, {'fig': 3, 'fiber': 'LS', 'span': 5, 'curve': 'Code GN', 'eta_db': 43.314028894058765}, {'fig': 3, 'fiber': 'LS', 'span': 5, 'curve': 'Code EGN', 'eta_db': 38.93129742371997}, {'fig': 6, 'fiber': 'LS', 'span': 5, 'curve': 'Code GN', 'eta_db': 43.314028894058765}, {'fig': 6, 'fiber': 'LS', 'span': 5, 'curve': 'Code EGN', 'eta_db': 40.21048671173983}, {'fig': 3, 'fiber': 'LS', 'span': 10, 'curve': 'Code GN', 'eta_db': 46.38234539941279}, {'fig': 3, 'fiber': 'LS', 'span': 10, 'curve': 'Code EGN', 'eta_db': 41.970648838127325}, {'fig': 6, 'fiber': 'LS', 'span': 10, 'curve': 'Code GN', 'eta_db': 46.38234539941279}, {'fig': 6, 'fiber': 'LS', 'span': 10, 'curve': 'Code EGN', 'eta_db': 43.260358391443376}, {'fig': 3, 'fiber': 'LS', 'span': 20, 'curve': 'Code GN', 'eta_db': 49.442091052292895}, {'fig': 3, 'fiber': 'LS', 'span': 20, 'curve': 'Code EGN', 'eta_db': 45.03222087937406}, {'fig': 6, 'fiber': 'LS', 'span': 20, 'curve': 'Code GN', 'eta_db': 49.442091052292895}, {'fig': 6, 'fiber': 'LS', 'span': 20, 'curve': 'Code EGN', 'eta_db': 46.31490473981983}, {'fig': 3, 'fiber': 'LS', 'span': 50, 'curve': 'Code GN', 'eta_db': 53.52255258178472}, {'fig': 3, 'fiber': 'LS', 'span': 50, 'curve': 'Code EGN', 'eta_db': 49.60501317108145}, {'fig': 6, 'fiber': 'LS', 'span': 50, 'curve': 'Code GN', 'eta_db': 53.52255258178472}, {'fig': 6, 'fiber': 'LS', 'span': 50, 'curve': 'Code EGN', 'eta_db': 50.77040226472404}, {'fig': 8, 'fiber': 'SMF', 'span': 1, 'curve': 'Code GN', 'eta_db': 27.702654135662467}, {'fig': 8, 'fiber': 'SMF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 22.268443811896297}, {'fig': 8, 'fiber': 'SMF', 'span': 2, 'curve': 'Code GN', 'eta_db': 30.735524777022874}, {'fig': 8, 'fiber': 'SMF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 27.39792439794932}, {'fig': 8, 'fiber': 'SMF', 'span': 5, 'curve': 'Code GN', 'eta_db': 34.717560934691974}, {'fig': 8, 'fiber': 'SMF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 33.00189353531648}, {'fig': 8, 'fiber': 'SMF', 'span': 10, 'curve': 'Code GN', 'eta_db': 37.70450397538386}, {'fig': 8, 'fiber': 'SMF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 36.76044450326987}, {'fig': 8, 'fiber': 'SMF', 'span': 20, 'curve': 'Code GN', 'eta_db': 40.69040067854839}, {'fig': 8, 'fiber': 'SMF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 40.27020122287904}, {'fig': 8, 'fiber': 'SMF', 'span': 50, 'curve': 'Code GN', 'eta_db': 44.41481312012377}, {'fig': 8, 'fiber': 'SMF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 44.2358807827228}, {'fig': 8, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code GN', 'eta_db': 34.66439492735915}, {'fig': 8, 'fiber': 'NZDSF', 'span': 1, 'curve': 'Code EGN', 'eta_db': 30.7540576112846}, {'fig': 8, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code GN', 'eta_db': 37.71604916223124}, {'fig': 8, 'fiber': 'NZDSF', 'span': 2, 'curve': 'Code EGN', 'eta_db': 34.556100268533704}, {'fig': 8, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code GN', 'eta_db': 41.73724248285247}, {'fig': 8, 'fiber': 'NZDSF', 'span': 5, 'curve': 'Code EGN', 'eta_db': 39.43872230313248}, {'fig': 8, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code GN', 'eta_db': 44.77868922304754}, {'fig': 8, 'fiber': 'NZDSF', 'span': 10, 'curve': 'Code EGN', 'eta_db': 42.78293675369102}, {'fig': 8, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code GN', 'eta_db': 47.8172044831999}, {'fig': 8, 'fiber': 'NZDSF', 'span': 20, 'curve': 'Code EGN', 'eta_db': 46.343128930961385}, {'fig': 8, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code GN', 'eta_db': 51.871455241481726}, {'fig': 8, 'fiber': 'NZDSF', 'span': 50, 'curve': 'Code EGN', 'eta_db': 51.16075626922356}, {'fig': 8, 'fiber': 'LS', 'span': 1, 'curve': 'Code GN', 'eta_db': 40.57993703811303}, {'fig': 8, 'fiber': 'LS', 'span': 1, 'curve': 'Code EGN', 'eta_db': 37.56737395987045}, {'fig': 8, 'fiber': 'LS', 'span': 2, 'curve': 'Code GN', 'eta_db': 43.66869234593632}, {'fig': 8, 'fiber': 'LS', 'span': 2, 'curve': 'Code EGN', 'eta_db': 40.70117484022792}, {'fig': 8, 'fiber': 'LS', 'span': 5, 'curve': 'Code GN', 'eta_db': 47.72593646937529}, {'fig': 8, 'fiber': 'LS', 'span': 5, 'curve': 'Code EGN', 'eta_db': 45.20812669055941}, {'fig': 8, 'fiber': 'LS', 'span': 10, 'curve': 'Code GN', 'eta_db': 50.79092701700394}, {'fig': 8, 'fiber': 'LS', 'span': 10, 'curve': 'Code EGN', 'eta_db': 48.51879333654223}, {'fig': 8, 'fiber': 'LS', 'span': 20, 'curve': 'Code GN', 'eta_db': 53.83623408365544}, {'fig': 8, 'fiber': 'LS', 'span': 20, 'curve': 'Code EGN', 'eta_db': 51.7134788153674}, {'fig': 8, 'fiber': 'LS', 'span': 50, 'curve': 'Code GN', 'eta_db': 57.82260845005791}, {'fig': 8, 'fiber': 'LS', 'span': 50, 'curve': 'Code EGN', 'eta_db': 56.36690992265181}])
print('paper rows:', len(paper), 'code rows:', len(code))
paper.head()

## 3. Paper vs Code 그래프

In [ ]:

fig=1; fiber="SMF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 1 — SMF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=1; fiber="NZDSF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 1 — NZDSF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=1; fiber="LS"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 1 — LS: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=3; fiber="SMF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 3 — SMF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=3; fiber="NZDSF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 3 — NZDSF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=3; fiber="LS"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 3 — LS: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=6; fiber="SMF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 6 — SMF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=6; fiber="NZDSF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 6 — NZDSF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=6; fiber="LS"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 6 — LS: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=8; fiber="SMF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 8 — SMF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=8; fiber="NZDSF"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 8 — NZDSF: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


In [ ]:

fig=8; fiber="LS"
plt.figure(figsize=(7.4,4.8))
for curve in ["GN","EGN","Simulation"]:
    d=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="o",label=f"Paper {curve}")
for curve in ["Code GN","Code EGN"]:
    d=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==curve)].sort_values("span")
    if len(d):
        plt.plot(d.span,d.eta_db,marker="s",label=curve)
plt.xscale("log")
plt.xticks([1,2,5,10,20,50],[1,2,5,10,20,50])
plt.xlabel("Number of spans")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Carena Fig. 8 — LS: Paper vs final_EGN.py")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()


## 4. 오차 계산

In [ ]:

details=[]
summaries=[]
for fig in [1,3,6,8]:
    for fiber in ["SMF","NZDSF","LS"]:
        for code_curve,paper_curve,label in [
            ("Code GN","GN","GN vs paper GN"),
            ("Code EGN","EGN","EGN vs paper EGN"),
            ("Code EGN","Simulation","EGN vs paper simulation"),
        ]:
            c=code[(code.fig==fig)&(code.fiber==fiber)&(code.curve==code_curve)][["span","eta_db"]].rename(columns={"eta_db":"code_db"})
            p=paper[(paper.fig==fig)&(paper.fiber==fiber)&(paper.curve==paper_curve)][["span","eta_db"]].rename(columns={"eta_db":"paper_db"})
            if c.empty or p.empty:
                continue
            m=c.merge(p,on="span")
            m["error_db"]=m.code_db-m.paper_db
            m["abs_error_db"]=m.error_db.abs()
            m["relative_error_pct"]=(10**(m.code_db/10)-10**(m.paper_db/10)).abs()/10**(m.paper_db/10)*100
            m["fig"]=fig; m["fiber"]=fiber; m["comparison"]=label
            details.append(m)
            summaries.append({
                "Figure":fig,"Fiber":fiber,"Comparison":label,
                "MAE_dB":m.abs_error_db.mean(),
                "Max_abs_error_dB":m.abs_error_db.max(),
                "MAPE_pct_linear_eta":m.relative_error_pct.mean(),
                "Max_relative_error_pct":m.relative_error_pct.max(),
            })
detail_df=pd.concat(details,ignore_index=True)
summary_df=pd.DataFrame(summaries)
summary_df.round(3)


### Figure별 전체 요약

|   Figure | Comparison              |   MAE_dB |   Max_abs_error_dB |   MAPE_pct_linear_eta |   Max_relative_error_pct |
|---------:|:------------------------|---------:|-------------------:|----------------------:|-------------------------:|
|        1 | GN vs paper GN          |    0.078 |              0.171 |                 1.769 |                    3.872 |
|        1 | EGN vs paper EGN        |    0.144 |              0.846 |                 3.452 |                   21.497 |
|        1 | EGN vs paper simulation |    1.233 |              4.292 |                40.328 |                  168.631 |
|        3 | GN vs paper GN          |    0.045 |              0.118 |                 1.034 |                    2.686 |
|        3 | EGN vs paper EGN        |    0.331 |              1.491 |                 8.412 |                   40.959 |
|        3 | EGN vs paper simulation |    0.678 |              1.118 |                14.764 |                   29.365 |
|        6 | GN vs paper GN          |    0.047 |              0.118 |                 1.086 |                    2.759 |
|        6 | EGN vs paper EGN        |    0.296 |              1.374 |                 7.451 |                   37.214 |
|        6 | EGN vs paper simulation |    0.298 |              1.194 |                 7.383 |                   31.635 |
|        8 | GN vs paper GN          |    0.072 |              0.346 |                 1.642 |                    7.652 |
|        8 | EGN vs paper EGN        |    0.462 |              1.313 |                11.576 |                   35.286 |
|        8 | EGN vs paper simulation |    0.416 |              1.221 |                10.49  |                   32.479 |


### 해석
- `GN vs paper GN`은 Fig. 1/3/6/8 모두 매우 작은 오차를 보입니다. 즉 **Poggiolini GN baseline 구현 자체는 신뢰도가 높습니다.**
- Fig. 1의 `Code EGN vs Paper EGN`도 전반적으로 잘 맞습니다. 특히 50-span에서는 거의 일치합니다.
- Fig. 3/6/8의 Full-EGN은 일부 조건에서 1 dB 이상의 차이가 발생합니다. 특히 SMF의 장거리와 9-channel 조건에서 두드러집니다.
- 단순히 이 숫자만 보면 “중간 정도로 잘 맞는다”고 볼 수 있지만, 다음 convergence test가 더 중요합니다.


## 5. Full EGN quadrature convergence 진단

In [ ]:
conv = pd.DataFrame([{'channels': 3, 'frequency_order': 32, 'xci_correction_W': -4.525162572977069e-07, 'mci_correction_W': -3.19861865326288e-09, 'result_eta_db': 41.75632111179156, 'paper_eta_db': 40.28754802916961}, {'channels': 3, 'frequency_order': 48, 'xci_correction_W': -1.0185410307612022e-05, 'mci_correction_W': -2.9258831958740847e-09, 'result_eta_db': 37.20285739914913, 'paper_eta_db': 40.28754802916961}, {'channels': 3, 'frequency_order': 64, 'xci_correction_W': -7.766979065066592e-07, 'mci_correction_W': -2.523687507229472e-09, 'result_eta_db': 41.661530259975436, 'paper_eta_db': 40.28754802916961}, {'channels': 3, 'frequency_order': 80, 'xci_correction_W': -2.602996016642429e-06, 'mci_correction_W': -2.7413791004591204e-09, 'result_eta_db': 41.08366347909852, 'paper_eta_db': 40.28754802916961}, {'channels': 3, 'frequency_order': 96, 'xci_correction_W': -1.3476640178927499e-05, 'mci_correction_W': -2.1946204666735625e-09, 'result_eta_db': 32.92484248826864, 'paper_eta_db': 40.28754802916961}, {'channels': 9, 'frequency_order': 32, 'xci_correction_W': -7.957529099288824e-07, 'mci_correction_W': -1.7419893381991215e-08, 'result_eta_db': 44.28510865489717, 'paper_eta_db': 43.12799441833452}, {'channels': 9, 'frequency_order': 48, 'xci_correction_W': -3.17840748264845e-05, 'mci_correction_W': -1.0330610785326747e-08, 'result_eta_db': nan, 'paper_eta_db': 43.12799441833452}, {'channels': 9, 'frequency_order': 64, 'xci_correction_W': -1.1050496422437971e-06, 'mci_correction_W': -1.045128550925075e-08, 'result_eta_db': 44.23588078272279, 'paper_eta_db': 43.12799441833452}, {'channels': 9, 'frequency_order': 80, 'xci_correction_W': -3.3440231708462266e-06, 'mci_correction_W': -9.46027015220255e-09, 'result_eta_db': 43.853006678545505, 'paper_eta_db': 43.12799441833452}, {'channels': 9, 'frequency_order': 96, 'xci_correction_W': -1.8265434548407347e-05, 'mci_correction_W': -8.552856674708448e-09, 'result_eta_db': 39.7138731023322, 'paper_eta_db': 43.12799441833452}])
conv.round(6)

In [ ]:

plt.figure(figsize=(7.4,4.8))
for nch in [3,9]:
    d=conv[conv.channels==nch].sort_values("frequency_order")
    plt.plot(d.frequency_order,d.result_eta_db,marker="o",label=f"Code {nch}-channel")
    plt.plot(d.frequency_order,d.paper_eta_db,marker="s",label=f"Paper target {nch}-channel")
plt.xlabel("EGN frequency_order")
plt.ylabel(r"$\eta$ [dB(1/W$^2$)]")
plt.title("Quadrature-order sensitivity — SMF, 50 spans")
plt.grid(True,alpha=0.3)
plt.legend()
plt.show()



### Convergence 결과가 의미하는 것

현재 XCI/MCI correction 경로는 `frequency_order`를 32→48→64→80→96으로 바꿀 때 결과가 단조롭게 수렴하지 않습니다.
특히 SMF 50-span에서 일부 order는 매우 큰 correction을 발생시키며, 9-channel에서는 특정 order에서 물리적으로 허용하기 어려운 결과(총 XMCI가 0 이하가 되는 경우)도 나타납니다.

따라서:
1. **Fig. 3/6/8의 order=64 결과가 paper와 비교적 가까워 보여도 이를 converged result로 간주하면 안 됩니다.**
2. 오차의 주원인은 단순 모델 구조가 아니라, 현재 Appendix-C factorized oscillatory integral을 fixed Gauss–Legendre quadrature로 평가하는 수치 방식의 안정성 문제일 가능성이 큽니다.
3. XCI/MCI correction에 대해 phase-aware interval splitting / adaptive quadrature / oscillatory quadrature 또는 충분한 검증된 change-of-variable가 필요합니다.



## 6. 최종 의견 — 다른 실험용 시뮬레이션에 사용할 수 있는가?

### A. GN baseline
**사용 가능: Yes.**
- Paper GN과의 오차가 매우 작습니다.
- 동일한 Poggiolini GN 가정 범위(dual-pol Manakov, uncompensated coherent link 등)에서는 다른 fiber/WDM 조건의 baseline NLI 계산에 충분히 유용합니다.

### B. SCI-EGN
**조건부 사용 가능: Yes, research-grade.**
- Fig. 1 paper EGN을 잘 재현합니다.
- rectangular-spectrum / coherent accumulation 조건이 맞는 경우 SCI modulation correction 연구에 활용할 수 있습니다.
- SSFM curve와 초기 span에서 차이가 있는 것은 paper 자체에서도 관찰되는 현상입니다.

### C. Full EGN (XCI + MCI)
**현재 버전으로 정량 sign-off: No.**
- Fig. 3/6/8 paper 곡선과 한 setting에서는 수 dB가 아닌 대체로 0.x~1.x dB 수준으로 접근하지만,
- quadrature order에 따라 correction 결과가 크게 변하므로 **수치 수렴성이 확보되지 않았습니다.**
- 따라서 새로운 fiber, modulation, 채널 수, 저분산 조건에서 “논문 수준 정확도”를 보장할 수 없습니다.

### 권장 다음 단계
1. XCI/MCI reduced integral의 식별/normalization을 Appendix A/B/C 원식과 항별로 다시 대조
2. fixed Gauss-Legendre 대신 oscillatory phase에 맞는 adaptive/split quadrature 구현
3. Fig. 3/6/8에서 모든 span에 대해 quadrature convergence를 먼저 확보
4. 그 후 독립 SSFM.py로 paper simulation과 재검증
5. 최종적으로 modulation 변경(16QAM/64QAM 등) 및 다른 fiber 조건으로 외삽 검증

**현재 판정:** `final_EGN.py`는 **GN + SCI-EGN은 강한 기반**, Full-EGN XCI/MCI는 **유망하지만 아직 numerical verification 단계**입니다.



## 7. 재현용 설정

본 보고서의 실행 결과:
- Fig. 1 SCI: `egn_frequency_order=256`, receiver integration 21 points
- Fig. 3/6/8: Sobol `2^16`, seed=7, `frequency_order=64`, receiver points=5
- launch power: 0 dBm/channel (η normalization 때문에 결과는 P³로 정규화)
- fitting / empirical offset: **사용하지 않음**

Paper curve data는 PDF vector extraction 결과이며, 코드 결과는 첨부된 `final_EGN.py`를 직접 실행하여 생성했습니다.
